# Text models (choose experiment and text representation)

In [ ]:
# ======================================================================
# SCRIPT 3: STRUCTURED-ONLY ML BASELINE + TEXTUAL REPRESENTATIONS ADDED
# ======================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Experiment choice
# ------------------------------------------------------------
# Choose one:
# "opinion_with_odds"   = structured + opinion text
# "factual_with_odds"   = structured + engineered factual features

EXPERIMENT = "opinion_with_odds"

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/Speciale"

PATH_WITH_ODDS = os.path.join(BASE_DIR, "final_with_odds_dataset.csv")
PATH_NO_ODDS = os.path.join(BASE_DIR, "final_no_odds_dataset.csv")
PATH_SQUAD_MAPPING = os.path.join(BASE_DIR, "squad_mapping.xlsx")

OUTPUT_DIR = os.path.join(BASE_DIR, "text_model_final_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# Derived experiment settings
# ------------------------------------------------------------
USE_ODDS = "with_odds" in EXPERIMENT

USE_OPINION_TEXT = False
USE_FACTUAL_FEATURES = False

if "combined" in EXPERIMENT:
    TEXT_TYPE = "combined"
    USE_OPINION_TEXT = True
    USE_FACTUAL_FEATURES = True
    TEXT_COLUMN = "opinion_text"

elif "opinion" in EXPERIMENT:
    TEXT_TYPE = "opinion"
    USE_OPINION_TEXT = True
    USE_FACTUAL_FEATURES = False
    TEXT_COLUMN = "opinion_text"

elif "factual" in EXPERIMENT:
    TEXT_TYPE = "factual"
    USE_OPINION_TEXT = False
    USE_FACTUAL_FEATURES = True
    TEXT_COLUMN = None

else:
    raise ValueError("EXPERIMENT must contain 'opinion', 'factual', or 'combined'.")

DATA_PATH = PATH_WITH_ODDS if USE_ODDS else PATH_NO_ODDS

print("Experiment:", EXPERIMENT)
print("Text type:", TEXT_TYPE)
print("Use odds:", USE_ODDS)
print("Use opinion text:", USE_OPINION_TEXT)
print("Use factual engineered features:", USE_FACTUAL_FEATURES)
print("Text column:", TEXT_COLUMN)
print("Data path:", DATA_PATH)
print("Squad mapping path:", PATH_SQUAD_MAPPING)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# BLOCK 2: LOAD DATA, CLEAN COLUMNS, DEFINE FEATURES AND TARGET
# ============================================================

# ------------------------------------------------------------
# 1. Load selected dataset
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

print("Loaded data shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. Basic column cleaning
# ------------------------------------------------------------
def clean_column_name(col):
    col = str(col)
    col = col.replace("\u00a0", "")
    col = col.replace("\n", "")
    col = col.replace("\t", "")
    col = col.strip()
    return col

df.columns = [clean_column_name(c) for c in df.columns]

# ------------------------------------------------------------
# 3. Parse date column
# ------------------------------------------------------------
df["MatchDatetime"] = pd.to_datetime(
    df["MatchDatetime"],
    errors="coerce",
    dayfirst=False
)

# ------------------------------------------------------------
# 4. Define target
# ------------------------------------------------------------
target_map = {
    "H": 0,
    "D": 1,
    "A": 2
}

target_names = {
    0: "Home win",
    1: "Draw",
    2: "Away win"
}

df["target"] = df["FTR"].map(target_map)

if df["target"].isna().any():
    raise ValueError("Some FTR values could not be mapped to target labels.")

# ------------------------------------------------------------
# 5. Define feature columns
# ------------------------------------------------------------
base_feature_cols = [
    "Home_attack",
    "Home_defence",
    "Away_attack",
    "Away_defence",
    "Home_PPG",
    "Away_PPG",
    "Home_GDpg",
    "Away_GDpg",
    "Home_form_PPG",
    "Away_form_PPG",
    "Attack_diff",
    "Defence_diff",
    "PPG_diff",
    "GDpg_diff",
    "Form_PPG_diff",
    "Interaction_home",
    "Interaction_away",
    "Strength_diff_abs"
]

odds_feature_cols = [
    "AvgH",
    "AvgD",
    "AvgA"
]

factual_feature_cols = [
    "home_n_injured",
    "away_n_injured",
    "home_n_doubtful",
    "away_n_doubtful",
    "home_n_suspended",
    "away_n_suspended",
    "home_n_unavailable",
    "away_n_unavailable",

    "home_form_points",
    "away_form_points",

    "home_top_scorer_goals",
    "away_top_scorer_goals",
    "home_top_scorer_missing",
    "away_top_scorer_missing",

    "diff_injured",
    "diff_doubtful",
    "diff_suspended",
    "diff_unavailable",
    "diff_form_points",
    "diff_top_scorer_goals"
]

structured_feature_cols = base_feature_cols.copy()

if USE_ODDS:
    structured_feature_cols += odds_feature_cols

if USE_FACTUAL_FEATURES:
    structured_feature_cols += factual_feature_cols

# ------------------------------------------------------------
# 6. Check required feature columns
# ------------------------------------------------------------
missing_features = [col for col in structured_feature_cols if col not in df.columns]

if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")

# ------------------------------------------------------------
# 7. Clean numeric columns
# ------------------------------------------------------------
def clean_numeric_column(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace({"-": np.nan, "": np.nan, "nan": np.nan, "None": np.nan})
        .astype(float)
    )

for col in structured_feature_cols:
    df[col] = clean_numeric_column(df[col])

# ------------------------------------------------------------
# 8. Text sanity checks
# ------------------------------------------------------------
text_essential_cols = []

if USE_OPINION_TEXT:
    if TEXT_COLUMN not in df.columns:
        raise ValueError(f"Missing selected text column: {TEXT_COLUMN}")

    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str).str.strip()

    blank_text_rows = (df[TEXT_COLUMN] == "").sum()

    print(f"Blank rows in {TEXT_COLUMN}: {blank_text_rows}")

    if blank_text_rows > 0:
        print("Dropping rows with blank selected text.")
        df = df[df[TEXT_COLUMN] != ""].copy()

    text_essential_cols = [TEXT_COLUMN]

else:
    print("No text column required for this experiment.")

# ------------------------------------------------------------
# 9. Drop rows with missing essential values
# ------------------------------------------------------------
essential_cols = [
    "MatchDatetime",
    "season",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "target"
] + text_essential_cols + structured_feature_cols

before_drop = df.shape[0]

df = df.dropna(subset=essential_cols).copy()

after_drop = df.shape[0]

print("Rows before missing-value drop:", before_drop)
print("Rows after missing-value drop:", after_drop)

# ------------------------------------------------------------
# 10. Sort and reset index
# ------------------------------------------------------------
df = df.sort_values("MatchDatetime").reset_index(drop=True)

# ------------------------------------------------------------
# 11. Final checks
# ------------------------------------------------------------
print("\nFinal dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["target"].map(target_names).value_counts())

print("\nSelected structured/numeric features:")
print(structured_feature_cols)

print("\nUse odds:")
print(USE_ODDS)

print("\nUse factual engineered features:")
print(USE_FACTUAL_FEATURES)

print("\nUse opinion text:")
print(USE_OPINION_TEXT)

print("\nText column(s):")
print(text_essential_cols)

display(df.head())

In [ ]:
# ============================================================
# BLOCK 3: SQUAD MAPPING, TEAM ALIASES AND PLAYER ALIASES
# ============================================================

if not USE_OPINION_TEXT:
    print("Skipping Block 3: no opinion text used in this experiment.")

else:

  # ------------------------------------------------------------
  # 1. Load squad mapping
  # ------------------------------------------------------------
  squad_df = pd.read_excel(PATH_SQUAD_MAPPING)

  squad_df.columns = [clean_column_name(c) for c in squad_df.columns]

  required_squad_cols = ["Team", "Player", "Season"]
  missing_squad_cols = [col for col in required_squad_cols if col not in squad_df.columns]

  if missing_squad_cols:
      raise ValueError(f"Missing columns in squad mapping: {missing_squad_cols}")

  squad_df = squad_df[required_squad_cols].copy()

  squad_df["Team"] = squad_df["Team"].astype(str).str.strip()
  squad_df["Player"] = squad_df["Player"].astype(str).str.strip()
  squad_df["Season"] = squad_df["Season"].astype(str).str.strip()

  print("Squad mapping shape:", squad_df.shape)
  display(squad_df.head())

  # ------------------------------------------------------------
  # 2. Standardise season format
  # ------------------------------------------------------------
  def standardise_season(season):
      season = str(season).strip()

      if re.match(r"^\d{4}_\d{4}$", season):
          return season

      if re.match(r"^\d{2}/\d{2}$", season):
          start = int(season[:2])
          end = int(season[-2:])

          start_year = 2000 + start
          end_year = 2000 + end

          return f"{start_year}_{end_year}"

      return season

  squad_df["season"] = squad_df["Season"].apply(standardise_season)

  # ------------------------------------------------------------
  # 3. Harmonise team names
  # ------------------------------------------------------------
  team_name_map = {
      "Manchester United": "Man United",
      "Manchester City": "Man City",
      "Tottenham Hotspur": "Tottenham",
      "Nottingham Forest": "Nott'm Forest",
      "Wolverhampton Wanderers": "Wolves",
      "Brighton & Hove Albion": "Brighton",
      "West Ham United": "West Ham",
      "Leicester City": "Leicester",
      "Leeds United": "Leeds",
      "AFC Bournemouth": "Bournemouth",
      "Newcastle United": "Newcastle",
      "Aston Villa": "Aston Villa",
      "Crystal Palace": "Crystal Palace"
  }

  squad_df["Team"] = squad_df["Team"].replace(team_name_map)

  # ------------------------------------------------------------
  # 4. Text normalisation
  # ------------------------------------------------------------
  def normalise_text(text):
      text = str(text).lower()
      text = text.replace("’", "'")
      text = text.replace("‘", "'")
      text = text.replace("“", '"')
      text = text.replace("”", '"')
      text = re.sub(r"\s+", " ", text)
      return text.strip()

  # ------------------------------------------------------------
  # 5. Player alias creation
  # ------------------------------------------------------------
  def create_player_aliases(player_name):
      player_name = normalise_text(player_name)

      if player_name in ["", "nan", "none"]:
          return []

      parts = player_name.split()

      aliases = set()
      aliases.add(player_name)

      # Add surname only
      if len(parts) >= 2:
          aliases.add(parts[-1])

      aliases = {
          a for a in aliases
          if len(a) >= 4
          and a not in [
              "none", "goal", "form", "city", "club", "team", "side",
              "james", "smith", "white", "young", "long", "ward", "gray",
              "cook", "rice", "browne", "roberts"
          ]
      }

      return sorted(aliases)

  # ------------------------------------------------------------
  # 6. Team aliases
  # ------------------------------------------------------------
  TEAM_STRONG_ALIASES = {
      "Arsenal": [
          "arsenal", "gunners", "mikel arteta", "arteta"
      ],
      "Aston Villa": [
          "aston villa", "villa", "villans", "unai emery", "emery", "steven gerrard", "gerrard"
      ],
      "Bournemouth": [
          "bournemouth", "cherries", "the cherries", "andoni iraola", "iraola", "scott parker", "parker"
      ],
      "Brentford": [
          "brentford", "bees", "the bees", "thomas frank"
      ],
      "Brighton": [
          "brighton", "seagulls", "the seagulls", "roberto de zerbi", "de zerbi",
          "graham potter", "potter", "fabian hurzeler", "hürzeler", "hurzeler"
      ],
      "Chelsea": [
          "chelsea", "blues", "the blues", "thomas tuchel", "tuchel",
          "mauricio pochettino", "pochettino", "enzo maresca", "maresca"
      ],
      "Crystal Palace": [
          "crystal palace", "palace", "eagles", "the eagles", "patrick vieira", "vieira",
          "roy hodgson", "hodgson", "oliver glasner", "glasner"
      ],
      "Everton": [
          "everton", "toffees", "the toffees", "frank lampard", "lampard", "sean dyche", "dyche"
      ],
      "Fulham": [
          "fulham", "cottagers", "the cottagers", "marco silva"
      ],
      "Leeds": [
          "leeds", "leeds united", "jesse marsch", "marsch", "javi gracia", "gracia", "daniel farke", "farke"
      ],
      "Leicester": [
          "leicester", "leicester city", "foxes", "the foxes", "brendan rodgers", "rodgers",
          "dean smith", "enzo maresca"
      ],
      "Liverpool": [
          "liverpool", "reds", "the reds", "jurgen klopp", "jürgen klopp", "klopp", "arne slot", "slot"
      ],
      "Man City": [
          "man city", "manchester city", "pep guardiola", "guardiola", "pep"
      ],
      "Man United": [
          "man united", "manchester united", "red devils", "the red devils",
          "erik ten hag", "ten hag", "ruben amorim", "rúben amorim", "amorim"
      ],
      "Newcastle": [
          "newcastle", "newcastle united", "magpies", "the magpies", "eddie howe", "howe"
      ],
      "Nott'm Forest": [
          "nott'm forest", "nottingham forest", "forest", "steve cooper", "cooper", "nuno espirito santo",
          "nuno espírito santo", "nuno"
      ],
      "Southampton": [
          "southampton", "saints", "the saints", "ralph hasenhuttl", "ralph hasenhüttl",
          "hasenhuttl", "hasenhüttl", "ruben selles", "rubén sellés", "selles", "sellés",
          "russell martin"
      ],
      "Tottenham": [
          "tottenham", "spurs", "antonio conte", "conte", "ange postecoglou", "postecoglou", "ange"
      ],
      "West Ham": [
          "west ham", "west ham united", "hammers", "the hammers", "david moyes", "moyes",
          "julen lopetegui", "lopetegui"
      ],
      "Wolves": [
          "wolves", "wolverhampton", "wolverhampton wanderers", "bruno lage", "lage",
          "julens lopetegui", "lopetegui", "gary o'neil", "oneil", "o'neil"
      ],
      "Burnley": [
          "burnley", "clarets", "the clarets", "vincent kompany", "kompany"
      ],
      "Luton": [
          "luton", "luton town", "hatters", "the hatters", "rob edwards"
      ],
      "Sheffield United": [
          "sheffield united", "sheffield utd", "blades", "the blades", "paul heckingbottom", "heckingbottom",
          "chris wilder", "wilder"
      ],
      "Ipswich": [
          "ipswich", "ipswich town", "tractor boys", "the tractor boys", "kieran mckenna", "mckenna"
      ]
  }

  TEAM_WEAK_ALIASES = {
      "Man City": ["city"],
      "Man United": ["united"],
      "Newcastle": ["united"],
      "Leeds": ["united"],
      "West Ham": ["united"],
      "Brighton": ["albion"],
      "Liverpool": ["reds"],
      "Nott'm Forest": ["reds"],
      "Chelsea": ["blues"],
      "Everton": ["blues"]
  }

  # ------------------------------------------------------------
  # 7. Ensure every team in data has an alias entry
  # ------------------------------------------------------------
  teams_in_data = sorted(set(df["HomeTeam"]).union(set(df["AwayTeam"])))

  for team in teams_in_data:
      if team not in TEAM_STRONG_ALIASES:
          TEAM_STRONG_ALIASES[team] = [team]

  print("\nTeams in data:")
  print(teams_in_data)

  print("\nTeams missing from original alias dictionary were added automatically if needed.")

  # ------------------------------------------------------------
  # 8. Build player lookup by team-season
  # ------------------------------------------------------------
  player_alias_lookup = {}

  for _, row in squad_df.iterrows():
      key = (row["Team"], row["season"])
      aliases = create_player_aliases(row["Player"])

      if key not in player_alias_lookup:
          player_alias_lookup[key] = set()

      player_alias_lookup[key].update(aliases)

  # Convert sets to sorted lists
  player_alias_lookup = {
      key: sorted(value)
      for key, value in player_alias_lookup.items()
  }

  print("\nNumber of team-season player alias groups:", len(player_alias_lookup))

  # ------------------------------------------------------------
  # 9. Row-specific alias retrieval
  # ------------------------------------------------------------
  def get_team_aliases(team, season):
      strong_aliases = set()
      weak_aliases = set()

      strong_aliases.update([normalise_text(a) for a in TEAM_STRONG_ALIASES.get(team, [team])])
      weak_aliases.update([normalise_text(a) for a in TEAM_WEAK_ALIASES.get(team, [])])

      player_aliases = player_alias_lookup.get((team, season), [])
      strong_aliases.update(player_aliases)

      strong_aliases = {
          a for a in strong_aliases
          if isinstance(a, str)
          and len(a) >= 3
          and a not in ["nan", "none", ""]
      }

      weak_aliases = {
          a for a in weak_aliases
          if isinstance(a, str)
          and len(a) >= 3
          and a not in ["nan", "none", ""]
      }

      return sorted(strong_aliases), sorted(weak_aliases)

  # ------------------------------------------------------------
  # 10. Test alias retrieval on first row
  # ------------------------------------------------------------
  example_row = df.iloc[0]

  home_strong, home_weak = get_team_aliases(
      example_row["HomeTeam"],
      example_row["season"]
  )

  away_strong, away_weak = get_team_aliases(
      example_row["AwayTeam"],
      example_row["season"]
  )

  print("\nExample match:")
  print(example_row["HomeTeam"], "vs", example_row["AwayTeam"], "-", example_row["season"])

  print("\nHome strong aliases example:")
  print(home_strong[:30])

  print("\nHome weak aliases example:")
  print(home_weak[:20])

  print("\nAway strong aliases example:")
  print(away_strong[:30])

  print("\nAway weak aliases example:")
  print(away_weak[:20])

In [ ]:
# ============================================================
# BLOCK 4: TEXT SPLITTING AND ATTRIBUTION
# Only used when opinion text is included.
# ============================================================

# ------------------------------------------------------------
# 1. Sentence splitter
# ------------------------------------------------------------
def split_sentences(text):
    text = normalise_text(text)
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]
    return sentences


# ------------------------------------------------------------
# 2. Context keywords
# ------------------------------------------------------------
HOME_CONTEXT = ["the hosts", "home side", "home team"]
AWAY_CONTEXT = ["the visitors", "away side", "away team"]


# ------------------------------------------------------------
# 3. Sentence scoring for opinion text
# ------------------------------------------------------------
def score_sentence(sentence, home_strong, home_weak, away_strong, away_weak):

    home_score = 0
    away_score = 0

    for alias in home_strong:
        if alias in sentence:
            home_score += 2

    for alias in away_strong:
        if alias in sentence:
            away_score += 2

    for alias in home_weak:
        if alias in sentence:
            home_score += 1

    for alias in away_weak:
        if alias in sentence:
            away_score += 1

    for phrase in HOME_CONTEXT:
        if phrase in sentence:
            home_score += 1

    for phrase in AWAY_CONTEXT:
        if phrase in sentence:
            away_score += 1

    return home_score, away_score


# ------------------------------------------------------------
# 4. Opinion text attribution
# ------------------------------------------------------------
def attribute_opinion_text(row):

    text = row["opinion_text"]
    sentences = split_sentences(text)

    home_strong, home_weak = get_team_aliases(row["HomeTeam"], row["season"])
    away_strong, away_weak = get_team_aliases(row["AwayTeam"], row["season"])

    home_sents = []
    away_sents = []
    both_sents = []
    neither_sents = []

    for sentence in sentences:

        home_score, away_score = score_sentence(
            sentence,
            home_strong,
            home_weak,
            away_strong,
            away_weak
        )

        if home_score == 0 and away_score == 0:
            neither_sents.append(sentence)

        elif home_score > 0 and away_score == 0:
            home_sents.append(sentence)

        elif away_score > 0 and home_score == 0:
            away_sents.append(sentence)

        else:
            both_sents.append(sentence)

    return pd.Series({
        "home_opinion_text": " ".join(home_sents),
        "away_opinion_text": " ".join(away_sents),
        "both_opinion_text": " ".join(both_sents),
        "neither_opinion_text": " ".join(neither_sents),
        "n_home": len(home_sents),
        "n_away": len(away_sents),
        "n_both": len(both_sents),
        "n_neither": len(neither_sents)
    })


# ------------------------------------------------------------
# 5. Run opinion text processing only if needed
# ------------------------------------------------------------
if USE_OPINION_TEXT:

    print("\nRunning opinion text attribution...")

    opinion_split = df.apply(attribute_opinion_text, axis=1)
    df = pd.concat([df, opinion_split], axis=1)

    df["home_text_final"] = (
        df["home_opinion_text"].fillna("").astype(str)
        + " "
        + df["both_opinion_text"].fillna("").astype(str)
    ).str.strip()

    df["away_text_final"] = (
        df["away_opinion_text"].fillna("").astype(str)
        + " "
        + df["both_opinion_text"].fillna("").astype(str)
    ).str.strip()

else:

    print("\nSkipping opinion text attribution: no opinion text used in this experiment.")

    df["home_opinion_text"] = ""
    df["away_opinion_text"] = ""
    df["both_opinion_text"] = ""
    df["neither_opinion_text"] = ""
    df["n_home"] = 0
    df["n_away"] = 0
    df["n_both"] = 0
    df["n_neither"] = 0
    df["home_text_final"] = ""
    df["away_text_final"] = ""


# ------------------------------------------------------------
# 6. Diagnostics
# ------------------------------------------------------------
print("\n--- TEXT ATTRIBUTION DIAGNOSTICS ---")

if USE_OPINION_TEXT:

    print("\nAverage opinion sentences per category:")
    print(df[["n_home", "n_away", "n_both", "n_neither"]].mean())

    print("\n% rows with ZERO away-only opinion sentences:")
    print((df["n_away"] == 0).mean())

    print("\n% rows with ZERO home-only opinion sentences:")
    print((df["n_home"] == 0).mean())

    print("\n% rows with EMPTY away opinion final text:")
    print((df["away_text_final"].str.strip() == "").mean())

    print("\n% rows with EMPTY home opinion final text:")
    print((df["home_text_final"].str.strip() == "").mean())

else:

    print("\nNo opinion text diagnostics needed.")


print("\nExample row:")
display(df[[
    "HomeTeam",
    "AwayTeam",
    "home_text_final",
    "away_text_final"
]].head(1))

In [ ]:
# ============================================================
# BLOCK 5: SIMPLE TEXT REPRESENTATIONS
# BAG OF WORDS / SENTIMENT
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    log_loss,
    recall_score,
    confusion_matrix
)
from scipy.sparse import hstack, csr_matrix
import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# 1. Choose text representation
# ------------------------------------------------------------
# Choose one:
# "bow"
# "sentiment"

TEXT_REPRESENTATION = "sentiment"

print("Experiment:", EXPERIMENT)
print("Text type:", TEXT_TYPE)
print("Use opinion text:", USE_OPINION_TEXT)
print("Use factual engineered features:", USE_FACTUAL_FEATURES)
print("Text representation:", TEXT_REPRESENTATION)

# ------------------------------------------------------------
# 2. Prepare text columns only when opinion text is used
# ------------------------------------------------------------
if USE_OPINION_TEXT:

    HOME_TEXT_COL = "home_text_final"
    AWAY_TEXT_COL = "away_text_final"

    print("Home text column:", HOME_TEXT_COL)
    print("Away text column:", AWAY_TEXT_COL)

    for col in [HOME_TEXT_COL, AWAY_TEXT_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column in df: {col}")

    home_texts = df[HOME_TEXT_COL].fillna("").astype(str)
    away_texts = df[AWAY_TEXT_COL].fillna("").astype(str)

else:

    HOME_TEXT_COL = None
    AWAY_TEXT_COL = None
    home_texts = None
    away_texts = None

    print("No opinion text used. Text representation will be skipped.")


# ------------------------------------------------------------
# 3. Create sentiment features only if opinion text exists
# ------------------------------------------------------------
positive_words = [
    "good", "great", "excellent", "strong", "impressive", "clinical",
    "confident", "dominant", "dangerous", "effective", "promising",
    "outstanding", "successful", "victory", "win", "wins", "winning",
    "boost", "fit", "available", "return", "returns", "fresh"
]

negative_words = [
    "poor", "weak", "bad", "struggling", "struggle", "injured",
    "injury", "injuries", "doubtful", "suspended", "absence",
    "absences", "missing", "defeat", "defeats", "lost", "loss",
    "collapse", "troubled", "desperate", "worst", "unavailable",
    "winless"
]


def sentiment_features(text_series):
    rows = []

    for text in text_series:
        text_lower = text.lower()
        tokens = text_lower.split()

        n_tokens = max(len(tokens), 1)

        pos_count = sum(text_lower.count(word) for word in positive_words)
        neg_count = sum(text_lower.count(word) for word in negative_words)

        rows.append([
            pos_count,
            neg_count,
            pos_count - neg_count,
            pos_count / n_tokens,
            neg_count / n_tokens,
            len(tokens)
        ])

    return np.array(rows)


if USE_OPINION_TEXT:

    home_sent = sentiment_features(home_texts)
    away_sent = sentiment_features(away_texts)

    X_sentiment = np.hstack([
        home_sent,
        away_sent,
        home_sent - away_sent
    ])

    print("Sentiment matrix shape:", X_sentiment.shape)

else:

    X_sentiment = None
    print("Sentiment matrix skipped.")


# ------------------------------------------------------------
# 4. Create Bag of Words vectoriser only if opinion text exists
# ------------------------------------------------------------
if USE_OPINION_TEXT:

    if TEXT_REPRESENTATION == "tfidf":

        vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.80,
            max_features=250
        )

    elif TEXT_REPRESENTATION == "bow":

        vectorizer = CountVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.80,
            max_features=250
        )

    elif TEXT_REPRESENTATION == "sentiment":

        vectorizer = None

    else:
        raise ValueError("TEXT_REPRESENTATION must be 'tfidf', 'bow', or 'sentiment'.")

else:

    vectorizer = None

In [ ]:
# ============================================================
# BLOCK 6A: CREATE MATRICES AND STRATIFIED SPLIT (70/15/15)
# SIMPLE TEXT / ENGINEERED FACTUAL MODEL
# ============================================================

from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Structured features, target and row IDs
# ------------------------------------------------------------
X_structured = df[structured_feature_cols].values
y = df["target"].values

df["row_id"] = (
    df["MatchDatetime"].astype(str) + "_" +
    df["HomeTeam"].astype(str) + "_" +
    df["AwayTeam"].astype(str)
)

indices = np.arange(len(df))

print("Structured feature shape:", X_structured.shape)
print("Target shape:", y.shape)
print("Index shape:", indices.shape)

# ------------------------------------------------------------
# 2. First split: Train (70%) / Temp (30%)
# ------------------------------------------------------------
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 3. Second split: Validation (15%) / Test (15%)
# ------------------------------------------------------------
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# 4. Store split assignment in dataframe
# ------------------------------------------------------------
df["split"] = ""

df.loc[train_idx, "split"] = "train"
df.loc[val_idx, "split"] = "validation"
df.loc[test_idx, "split"] = "test"

# ------------------------------------------------------------
# 5. Split structured features
# ------------------------------------------------------------
X_train_struct = X_structured[train_idx]
X_val_struct = X_structured[val_idx]
X_test_struct = X_structured[test_idx]

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

# ------------------------------------------------------------
# 6. Scale structured / numeric features
# ------------------------------------------------------------
scaler_struct = StandardScaler()

X_train_struct_scaled = scaler_struct.fit_transform(X_train_struct)
X_val_struct_scaled = scaler_struct.transform(X_val_struct)
X_test_struct_scaled = scaler_struct.transform(X_test_struct)

# ------------------------------------------------------------
# 7. Build text features only if opinion text is used
# ------------------------------------------------------------
if USE_OPINION_TEXT:

    if TEXT_REPRESENTATION in ["tfidf", "bow"]:

        train_home_texts = home_texts.iloc[train_idx]
        val_home_texts = home_texts.iloc[val_idx]
        test_home_texts = home_texts.iloc[test_idx]

        train_away_texts = away_texts.iloc[train_idx]
        val_away_texts = away_texts.iloc[val_idx]
        test_away_texts = away_texts.iloc[test_idx]

        # Fit vocabulary on home + away training text only
        train_all_texts = pd.concat([train_home_texts, train_away_texts], axis=0)
        vectorizer.fit(train_all_texts)

        # Transform home and away separately
        X_train_home = vectorizer.transform(train_home_texts)
        X_val_home = vectorizer.transform(val_home_texts)
        X_test_home = vectorizer.transform(test_home_texts)

        X_train_away = vectorizer.transform(train_away_texts)
        X_val_away = vectorizer.transform(val_away_texts)
        X_test_away = vectorizer.transform(test_away_texts)

        # Combine horizontally: home text + away text
        X_train_text = hstack([X_train_home, X_train_away]).tocsr()
        X_val_text = hstack([X_val_home, X_val_away]).tocsr()
        X_test_text = hstack([X_test_home, X_test_away]).tocsr()

    elif TEXT_REPRESENTATION == "sentiment":

        X_train_text = csr_matrix(X_sentiment[train_idx])
        X_val_text = csr_matrix(X_sentiment[val_idx])
        X_test_text = csr_matrix(X_sentiment[test_idx])

    else:
        raise ValueError("Invalid TEXT_REPRESENTATION.")

    # Combine structured/numeric + text
    X_train = hstack([
        csr_matrix(X_train_struct_scaled),
        X_train_text
    ]).tocsr()

    X_val = hstack([
        csr_matrix(X_val_struct_scaled),
        X_val_text
    ]).tocsr()

    X_test = hstack([
        csr_matrix(X_test_struct_scaled),
        X_test_text
    ]).tocsr()

else:

    print("\nNo opinion text used. Using structured/numeric features only.")

    X_train_text = None
    X_val_text = None
    X_test_text = None

    X_train = csr_matrix(X_train_struct_scaled)
    X_val = csr_matrix(X_val_struct_scaled)
    X_test = csr_matrix(X_test_struct_scaled)

# ------------------------------------------------------------
# 8. Final checks
# ------------------------------------------------------------
print("\n--- SPLIT SHAPES ---")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\n--- TARGET DISTRIBUTION ---")
print("Train:", np.bincount(y_train, minlength=3))
print("Validation:", np.bincount(y_val, minlength=3))
print("Test:", np.bincount(y_test, minlength=3))

print("\n--- SPLIT COUNTS ---")
print(df["split"].value_counts())

print("\n--- TEXT MATRIX SHAPES ---")
if USE_OPINION_TEXT:
    print("Train text:", X_train_text.shape)
    print("Validation text:", X_val_text.shape)
    print("Test text:", X_test_text.shape)
else:
    print("No text matrix created.")

display(df[["row_id", "HomeTeam", "AwayTeam", "FTR", "target", "split"]].head())

In [ ]:
# ============================================================
# BLOCK 7: REGULARISED LOGISTIC REGRESSION
# SIMPLE TEXT / ENGINEERED FACTUAL REPRESENTATIONS
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    log_loss,
    recall_score,
    confusion_matrix
)

import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# 1. Model input label
# ------------------------------------------------------------
MODEL_INPUT_TYPE = TEXT_REPRESENTATION if USE_OPINION_TEXT else "numeric_only"

print("Experiment:", EXPERIMENT)
print("Text type:", TEXT_TYPE)
print("Use opinion text:", USE_OPINION_TEXT)
print("Use factual engineered features:", USE_FACTUAL_FEATURES)
print("Model input type:", MODEL_INPUT_TYPE)

# ------------------------------------------------------------
# 2. Evaluation functions
# ------------------------------------------------------------
def rps_multiclass(y_true, probs, n_classes=3):
    y_onehot = np.eye(n_classes)[y_true]
    cum_probs = np.cumsum(probs, axis=1)
    cum_true = np.cumsum(y_onehot, axis=1)
    return np.mean(np.sum((cum_probs - cum_true) ** 2, axis=1) / (n_classes - 1))


def expected_calibration_error(y_true, probs, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = predictions == y_true

    ece = 0.0
    bin_edges = np.linspace(0, 1, n_bins + 1)

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == n_bins - 1:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences >= lower) & (confidences < upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            acc_bin = np.mean(accuracies[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop_in_bin * abs(acc_bin - conf_bin)

    return ece


def evaluate_model(model, X, y):
    probs = model.predict_proba(X)
    preds = np.argmax(probs, axis=1)

    return {
        "Accuracy": accuracy_score(y, preds),
        "BalancedAccuracy": balanced_accuracy_score(y, preds),
        "LogLoss": log_loss(y, probs, labels=[0, 1, 2]),
        "RPS": rps_multiclass(y, probs),
        "ECE": expected_calibration_error(y, probs),
        "DrawRecall": recall_score(
            y,
            preds,
            labels=[1],
            average=None,
            zero_division=0
        )[0],
        "Pred_H": np.sum(preds == 0),
        "Pred_D": np.sum(preds == 1),
        "Pred_A": np.sum(preds == 2)
    }

# ------------------------------------------------------------
# 3. Model grid
# ------------------------------------------------------------
lr_grid = []

for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]:
    for class_weight in [None, "balanced"]:
        lr_grid.append({
            "name": f"LR_L2_C{C}_cw{class_weight}",
            "penalty": "l2",
            "solver": "lbfgs",
            "C": C,
            "class_weight": class_weight,
            "l1_ratio": None
        })

for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    for class_weight in [None, "balanced"]:
        lr_grid.append({
            "name": f"LR_L1_C{C}_cw{class_weight}",
            "penalty": "l1",
            "solver": "saga",
            "C": C,
            "class_weight": class_weight,
            "l1_ratio": None
        })

for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1]:
    for l1_ratio in [0.25, 0.5, 0.75]:
        for class_weight in [None, "balanced"]:
            lr_grid.append({
                "name": f"LR_EN_C{C}_l1{l1_ratio}_cw{class_weight}",
                "penalty": "elasticnet",
                "solver": "saga",
                "C": C,
                "class_weight": class_weight,
                "l1_ratio": l1_ratio
            })

# ------------------------------------------------------------
# 4. Train and validate
# ------------------------------------------------------------
results = []
models = {}

for params in lr_grid:

    model = LogisticRegression(
        penalty=params["penalty"],
        solver=params["solver"],
        C=params["C"],
        class_weight=params["class_weight"],
        l1_ratio=params["l1_ratio"],
        multi_class="multinomial",
        max_iter=5000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    train_metrics = evaluate_model(model, X_train, y_train)
    val_metrics = evaluate_model(model, X_val, y_val)

    results.append({
        "Experiment": EXPERIMENT,
        "TextType": TEXT_TYPE,
        "UseOdds": USE_ODDS,
        "UseOpinionText": USE_OPINION_TEXT,
        "UseFactualFeatures": USE_FACTUAL_FEATURES,
        "TextRepresentation": MODEL_INPUT_TYPE,
        "Model": params["name"],
        "Penalty": params["penalty"],
        "C": params["C"],
        "ClassWeight": params["class_weight"],
        "L1Ratio": params["l1_ratio"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainBalancedAccuracy": train_metrics["BalancedAccuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],
        "TrainRPS": train_metrics["RPS"],
        "TrainECE": train_metrics["ECE"],
        "TrainDrawRecall": train_metrics["DrawRecall"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    models[params["name"]] = model

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("ValLogLoss").reset_index(drop=True)

print("=== TOP 20 MODELS BY VALIDATION LOG LOSS ===")
display(results_df.head(20))

best_model_name = results_df.loc[0, "Model"]
best_model = models[best_model_name]

print("\nBest model by validation log loss:")
print(best_model_name)

# ------------------------------------------------------------
# 5. Test once
# ------------------------------------------------------------
test_metrics = evaluate_model(best_model, X_test, y_test)

print("\n=== FINAL TEST RESULTS ===")
for key, value in test_metrics.items():
    print(f"{key}: {value}")

test_probs = best_model.predict_proba(X_test)
test_preds = np.argmax(test_probs, axis=1)

print("\nPrediction distribution [H, D, A]:")
print(np.bincount(test_preds, minlength=3))

print("\nTrue distribution [H, D, A]:")
print(np.bincount(y_test, minlength=3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_preds, labels=[0, 1, 2]))

# ------------------------------------------------------------
# 6. Save validation results
# ------------------------------------------------------------
if not os.path.exists(OUTPUT_DIR):
    OUTPUT_DIR = "/content/gdrive/MyDrive/Speciale/text_model_final_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

results_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_regularised_lr_results.csv"
)

results_df.to_csv(results_path, index=False)

print("\nSaved validation results to:")
print(results_path)

# ------------------------------------------------------------
# 7. Save final test summary
# ------------------------------------------------------------
test_summary = {
    "Experiment": EXPERIMENT,
    "TextType": TEXT_TYPE,
    "UseOdds": USE_ODDS,
    "UseOpinionText": USE_OPINION_TEXT,
    "UseFactualFeatures": USE_FACTUAL_FEATURES,
    "TextRepresentation": MODEL_INPUT_TYPE,
    "BestModel": best_model_name,
    **test_metrics
}

test_summary_df = pd.DataFrame([test_summary])

test_summary_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_regularised_lr_test_summary.csv"
)

test_summary_df.to_csv(test_summary_path, index=False)

print("\nSaved test summary to:")
print(test_summary_path)

In [ ]:
# ============================================================
# BLOCK 7B: RANDOM FOREST AND LINEAR SVM
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd
import numpy as np
import os

# ============================================================
# RANDOM FOREST
# ============================================================

print("\n=== RANDOM FOREST ===")

rf_grid = [
    {"name": "RF_200_depth10", "n_estimators": 200, "max_depth": 10},
    {"name": "RF_300_depth10", "n_estimators": 300, "max_depth": 10},
    {"name": "RF_300_depth15", "n_estimators": 300, "max_depth": 15},
]

rf_results = []
rf_models = {}

for params in rf_grid:

    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    train_metrics = evaluate_model(model, X_train, y_train)
    val_metrics = evaluate_model(model, X_val, y_val)

    rf_results.append({
        "Experiment": EXPERIMENT,
        "TextType": TEXT_TYPE,
        "UseOdds": USE_ODDS,
        "UseOpinionText": USE_OPINION_TEXT,
        "UseFactualFeatures": USE_FACTUAL_FEATURES,
        "TextRepresentation": MODEL_INPUT_TYPE,
        "ModelFamily": "RandomForest",
        "Model": params["name"],
        "N_estimators": params["n_estimators"],
        "MaxDepth": params["max_depth"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainBalancedAccuracy": train_metrics["BalancedAccuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],
        "TrainRPS": train_metrics["RPS"],
        "TrainECE": train_metrics["ECE"],
        "TrainDrawRecall": train_metrics["DrawRecall"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    rf_models[params["name"]] = model

rf_results_df = pd.DataFrame(rf_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== RANDOM FOREST RESULTS BY VALIDATION LOG LOSS ===")
display(rf_results_df)

best_rf_name = rf_results_df.loc[0, "Model"]
best_rf_model = rf_models[best_rf_name]

print("\nBest Random Forest model by validation log loss:")
print(best_rf_name)

rf_test_metrics = evaluate_model(best_rf_model, X_test, y_test)
rf_test_probs = best_rf_model.predict_proba(X_test)
rf_test_preds = np.argmax(rf_test_probs, axis=1)

print("\n=== FINAL TEST RESULTS: RANDOM FOREST ===")
for key, value in rf_test_metrics.items():
    print(f"{key}: {value}")

print("\nPrediction distribution [H, D, A]:")
print(np.bincount(rf_test_preds, minlength=3))

print("\nTrue distribution [H, D, A]:")
print(np.bincount(y_test, minlength=3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, rf_test_preds, labels=[0, 1, 2]))

rf_validation_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_random_forest_validation_results.csv"
)

rf_results_df.to_csv(rf_validation_path, index=False)

rf_test_summary = {
    "Experiment": EXPERIMENT,
    "TextType": TEXT_TYPE,
    "UseOdds": USE_ODDS,
    "UseOpinionText": USE_OPINION_TEXT,
    "UseFactualFeatures": USE_FACTUAL_FEATURES,
    "TextRepresentation": MODEL_INPUT_TYPE,
    "BestModel": best_rf_name,
    **rf_test_metrics
}

rf_test_summary_df = pd.DataFrame([rf_test_summary])

rf_test_summary_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_random_forest_test_summary.csv"
)

rf_test_summary_df.to_csv(rf_test_summary_path, index=False)

print("\nSaved Random Forest validation results to:")
print(rf_validation_path)

print("\nSaved Random Forest test summary to:")
print(rf_test_summary_path)


# ============================================================
# LINEAR SVM WITH POST-HOC CALIBRATION
# ============================================================

print("\n=== LINEAR SVM WITH POST-HOC CALIBRATION ===")

svm_grid = [
    {"name": "SVM_C0.01_cwNone", "C": 0.01, "class_weight": None},
    {"name": "SVM_C0.03_cwNone", "C": 0.03, "class_weight": None},
    {"name": "SVM_C0.1_cwNone", "C": 0.1, "class_weight": None},
    {"name": "SVM_C0.1_cwBalanced", "C": 0.1, "class_weight": "balanced"},
    {"name": "SVM_C0.3_cwNone", "C": 0.3, "class_weight": None}
]

svm_results = []
svm_models = {}

for params in svm_grid:

    base_svm = LinearSVC(
        C=params["C"],
        class_weight=params["class_weight"],
        random_state=RANDOM_STATE,
        max_iter=10000
    )

    model = CalibratedClassifierCV(
        estimator=base_svm,
        method="sigmoid",
        cv=3
    )

    model.fit(X_train, y_train)

    train_metrics = evaluate_model(model, X_train, y_train)
    val_metrics = evaluate_model(model, X_val, y_val)

    svm_results.append({
        "Experiment": EXPERIMENT,
        "TextType": TEXT_TYPE,
        "UseOdds": USE_ODDS,
        "UseOpinionText": USE_OPINION_TEXT,
        "UseFactualFeatures": USE_FACTUAL_FEATURES,
        "TextRepresentation": MODEL_INPUT_TYPE,
        "ModelFamily": "LinearSVM_Calibrated",
        "Model": params["name"],
        "C": params["C"],
        "ClassWeight": params["class_weight"],
        "Calibration": "sigmoid_cv3",

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainBalancedAccuracy": train_metrics["BalancedAccuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],
        "TrainRPS": train_metrics["RPS"],
        "TrainECE": train_metrics["ECE"],
        "TrainDrawRecall": train_metrics["DrawRecall"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    svm_models[params["name"]] = model

svm_results_df = pd.DataFrame(svm_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== LINEAR SVM RESULTS BY VALIDATION LOG LOSS ===")
display(svm_results_df)

best_svm_name = svm_results_df.loc[0, "Model"]
best_svm_model = svm_models[best_svm_name]

print("\nBest Linear SVM model by validation log loss:")
print(best_svm_name)

svm_test_metrics = evaluate_model(best_svm_model, X_test, y_test)
svm_test_probs = best_svm_model.predict_proba(X_test)
svm_test_preds = np.argmax(svm_test_probs, axis=1)

print("\n=== FINAL TEST RESULTS: LINEAR SVM ===")
for key, value in svm_test_metrics.items():
    print(f"{key}: {value}")

print("\nPrediction distribution [H, D, A]:")
print(np.bincount(svm_test_preds, minlength=3))

print("\nTrue distribution [H, D, A]:")
print(np.bincount(y_test, minlength=3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, svm_test_preds, labels=[0, 1, 2]))

svm_validation_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_linear_svm_validation_results.csv"
)

svm_results_df.to_csv(svm_validation_path, index=False)

svm_test_summary = {
    "Experiment": EXPERIMENT,
    "TextType": TEXT_TYPE,
    "UseOdds": USE_ODDS,
    "UseOpinionText": USE_OPINION_TEXT,
    "UseFactualFeatures": USE_FACTUAL_FEATURES,
    "TextRepresentation": MODEL_INPUT_TYPE,
    "BestModel": best_svm_name,
    **svm_test_metrics
}

svm_test_summary_df = pd.DataFrame([svm_test_summary])

svm_test_summary_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_linear_svm_test_summary.csv"
)

svm_test_summary_df.to_csv(svm_test_summary_path, index=False)

print("\nSaved Linear SVM validation results to:")
print(svm_validation_path)

print("\nSaved Linear SVM test summary to:")
print(svm_test_summary_path)

In [ ]:
# ============================================================
# BLOCK 7C: XGBOOST
# ============================================================

from xgboost import XGBClassifier

print("\n=== XGBOOST ===")

xgb_grid = [
    {
        "name": "XGB_200_depth2_lr003",
        "n_estimators": 200,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0
    },
    {
        "name": "XGB_300_depth2_lr003_l1",
        "n_estimators": 300,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 5,
        "reg_lambda": 5.0,
        "reg_alpha": 0.5
    },
    {
        "name": "XGB_300_depth3_lr003",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },
    {
        "name": "XGB_300_depth3_lr005",
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "min_child_weight": 7,
        "reg_lambda": 10.0,
        "reg_alpha": 0.5
    }
]

xgb_results = []
xgb_models = {}

for params in xgb_grid:

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    train_metrics = evaluate_model(model, X_train, y_train)
    val_metrics = evaluate_model(model, X_val, y_val)

    xgb_results.append({
        "Experiment": EXPERIMENT,
        "TextType": TEXT_TYPE,
        "UseOdds": USE_ODDS,
        "UseOpinionText": USE_OPINION_TEXT,
        "UseFactualFeatures": USE_FACTUAL_FEATURES,
        "TextRepresentation": MODEL_INPUT_TYPE,
        "ModelFamily": "XGBoost",
        "Model": params["name"],

        "N_estimators": params["n_estimators"],
        "MaxDepth": params["max_depth"],
        "LearningRate": params["learning_rate"],
        "Subsample": params["subsample"],
        "ColsampleBytree": params["colsample_bytree"],
        "MinChildWeight": params["min_child_weight"],
        "RegLambda": params["reg_lambda"],
        "RegAlpha": params["reg_alpha"],

        "TrainAccuracy": train_metrics["Accuracy"],
        "TrainBalancedAccuracy": train_metrics["BalancedAccuracy"],
        "TrainLogLoss": train_metrics["LogLoss"],
        "TrainRPS": train_metrics["RPS"],
        "TrainECE": train_metrics["ECE"],
        "TrainDrawRecall": train_metrics["DrawRecall"],

        "ValAccuracy": val_metrics["Accuracy"],
        "ValBalancedAccuracy": val_metrics["BalancedAccuracy"],
        "ValLogLoss": val_metrics["LogLoss"],
        "ValRPS": val_metrics["RPS"],
        "ValECE": val_metrics["ECE"],
        "ValDrawRecall": val_metrics["DrawRecall"],
        "ValPred_H": val_metrics["Pred_H"],
        "ValPred_D": val_metrics["Pred_D"],
        "ValPred_A": val_metrics["Pred_A"]
    })

    xgb_models[params["name"]] = model

xgb_results_df = pd.DataFrame(xgb_results).sort_values("ValLogLoss").reset_index(drop=True)

print("\n=== XGBOOST RESULTS BY VALIDATION LOG LOSS ===")
display(xgb_results_df)

best_xgb_name = xgb_results_df.loc[0, "Model"]
best_xgb_model = xgb_models[best_xgb_name]

print("\nBest XGBoost model by validation log loss:")
print(best_xgb_name)

xgb_test_metrics = evaluate_model(best_xgb_model, X_test, y_test)

print("\n=== TEST RESULTS: XGBOOST BEST ===")
for k, v in xgb_test_metrics.items():
    print(f"{k}: {v}")

xgb_test_probs = best_xgb_model.predict_proba(X_test)
xgb_test_preds = np.argmax(xgb_test_probs, axis=1)

print("\nPrediction distribution [H, D, A]:")
print(np.bincount(xgb_test_preds, minlength=3))

print("\nTrue distribution [H, D, A]:")
print(np.bincount(y_test, minlength=3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, xgb_test_preds, labels=[0, 1, 2]))

xgb_results_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_xgboost_validation_results.csv"
)

xgb_results_df.to_csv(xgb_results_path, index=False)

xgb_test_summary = {
    "Experiment": EXPERIMENT,
    "TextType": TEXT_TYPE,
    "UseOdds": USE_ODDS,
    "UseOpinionText": USE_OPINION_TEXT,
    "UseFactualFeatures": USE_FACTUAL_FEATURES,
    "TextRepresentation": MODEL_INPUT_TYPE,
    "BestModel": best_xgb_name,
    **xgb_test_metrics
}

xgb_test_summary_df = pd.DataFrame([xgb_test_summary])

xgb_test_summary_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_xgboost_test_summary.csv"
)

xgb_test_summary_df.to_csv(xgb_test_summary_path, index=False)

print("\nSaved XGBoost validation results to:")
print(xgb_results_path)

print("\nSaved XGBoost test summary to:")
print(xgb_test_summary_path)

# Calibration plots

In [ ]:
# ------------------------------------------------------------
# 12. CALIBRATION PLOTS
# ------------------------------------------------------------

import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def plot_multiclass_calibration(y_true, model_probs_dict, target_names, output_dir, experiment):
    """
    Plot one-vs-rest calibration curves for each class.
    Each class gets its own plot comparing all models.
    """

    n_classes = len(target_names)

    for class_id in range(n_classes):

        plt.figure(figsize=(7, 6))

        # Perfect calibration line
        plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")

        y_binary = (y_true == class_id).astype(int)

        for model_name, probs in model_probs_dict.items():

            prob_class = probs[:, class_id]

            frac_pos, mean_pred = calibration_curve(
                y_binary,
                prob_class,
                n_bins=10,
                strategy="uniform"
            )

            plt.plot(
                mean_pred,
                frac_pos,
                marker="o",
                label=model_name
            )

        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed frequency")
        plt.title(f"Calibration plot: {target_names[class_id]}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        plot_path = os.path.join(
            output_dir,
            f"{experiment}_calibration_{target_names[class_id].replace(' ', '_').lower()}.png"
        )

        plt.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.show()

        print(f"Saved calibration plot to: {plot_path}")


model_probs_dict = {
    "Logistic Regression": test_probs
}

plot_multiclass_calibration(
    y_true=y_test,
    model_probs_dict=model_probs_dict,
    target_names=target_names,
    output_dir=OUTPUT_DIR,
    experiment=EXPERIMENT
)

In [ ]:
# ============================================================
# CALIBRATION TABLE
# ============================================================

import numpy as np
import pandas as pd

class_names = ["Home win", "Draw", "Away win"]

def make_calibration_table(y_true, probs, model_name="Opinion LR", n_bins=10):
    rows = []

    bins = np.linspace(0, 1, n_bins + 1)

    for class_id, class_name in enumerate(class_names):

        y_binary = (y_true == class_id).astype(int)
        pred_prob = probs[:, class_id]

        bin_id = np.digitize(pred_prob, bins, right=False) - 1
        bin_id = np.clip(bin_id, 0, n_bins - 1)

        for b in range(n_bins):
            mask = bin_id == b

            if mask.sum() == 0:
                continue

            mean_predicted_probability = pred_prob[mask].mean()
            true_observed_probability = y_binary[mask].mean()

            rows.append({
                "model": model_name,
                "outcome": class_name,
                "probability_bin": f"{bins[b]:.1f}-{bins[b+1]:.1f}",
                "n_matches": mask.sum(),
                "predicted_probability": mean_predicted_probability,
                "true_probability": true_observed_probability,
                "calibration_error": true_observed_probability - mean_predicted_probability,
                "absolute_calibration_error": abs(true_observed_probability - mean_predicted_probability)
            })

    return pd.DataFrame(rows)


calibration_table = make_calibration_table(
    y_true=y_test,
    probs=test_probs,
    model_name="Opinion-based LR",
    n_bins=10
)

calibration_table

# Draw diagnostic

In [ ]:
# ============================================================
# DIAGNOSTIC: PROBABILITY ANALYSIS
# ============================================================

probs = best_model.predict_proba(X_test)
preds = np.argmax(probs, axis=1)

df_analysis = pd.DataFrame({
    "true": y_test,
    "pred": preds,
    "prob_home": probs[:, 0],
    "prob_draw": probs[:, 1],
    "prob_away": probs[:, 2]
})

df_analysis["correct"] = df_analysis["true"] == df_analysis["pred"]

# Max probability (confidence)
df_analysis["confidence"] = probs.max(axis=1)

# Difference between top 2 probs (uncertainty measure)
sorted_probs = np.sort(probs, axis=1)
df_analysis["margin"] = sorted_probs[:, -1] - sorted_probs[:, -2]

df_analysis.head()

In [ ]:
print("\n=== CONFIDENCE ===")
print("Correct:", df_analysis[df_analysis["correct"]]["confidence"].mean())
print("Incorrect:", df_analysis[~df_analysis["correct"]]["confidence"].mean())

print("\n=== MARGIN (top2 difference) ===")
print("Correct:", df_analysis[df_analysis["correct"]]["margin"].mean())
print("Incorrect:", df_analysis[~df_analysis["correct"]]["margin"].mean())

In [ ]:
df_wrong = df_analysis[~df_analysis["correct"]]

print("\n=== TRUE CLASS DISTRIBUTION FOR WRONG PREDICTIONS ===")
print(df_wrong["true"].value_counts(normalize=True))

In [ ]:
print("\n=== AVERAGE PROBABILITIES (WRONG PREDICTIONS) ===")
print(df_wrong[["prob_home", "prob_draw", "prob_away"]].mean())

In [ ]:
df_analysis["predicted_class"] = df_analysis["pred"]

# Cases where draw probability is high but not predicted
almost_draws = df_analysis[
    (df_analysis["prob_draw"] > 0.27) &
    (df_analysis["pred"] != 1)
]

print("Number of 'almost draws':", almost_draws.shape[0])

almost_draws.head(10)

In [ ]:
# ============================================================
# ANALYSE TRUE DRAWS
# ============================================================

df_draws = df_analysis[df_analysis["true"] == 1].copy()

print("Number of true draws:", df_draws.shape[0])

display(df_draws.head(20))

In [ ]:
df_draws_sorted = df_draws.sort_values("prob_draw", ascending=False)

display(df_draws_sorted.head(20))

In [ ]:
print("\n=== DRAW ANALYSIS ===")

print("\nAverage probabilities (true draws):")
print(df_draws[["prob_home", "prob_draw", "prob_away"]].mean())

print("\nHow often was draw predicted correctly?")
print((df_draws["pred"] == 1).mean())

print("\nHow often was draw the SECOND highest probability?")
second_highest_draw = (
    (df_draws["prob_draw"] < df_draws[["prob_home", "prob_away"]].max(axis=1)) &
    (df_draws["prob_draw"] > df_draws[["prob_home", "prob_away"]].min(axis=1))
).mean()

print(second_highest_draw)

In [ ]:
def predict_draw_sensitive(probs):
    preds = []

    for p in probs:
        if p[1] > 0.28:   # change this, just an experiment
            preds.append(1)
        else:
            preds.append(np.argmax(p))

    return np.array(preds)

In [ ]:
new_preds = predict_draw_sensitive(probs)

print("Accuracy:", accuracy_score(y_test, new_preds))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, new_preds))
print("Draw Recall:", recall_score(y_test, new_preds, labels=[1], average=None, zero_division=0)[0])

print("Pred dist:", np.bincount(new_preds, minlength=3))
print(confusion_matrix(y_test, new_preds))

In [ ]:
# ============================================================
# BLOCK 8: LOGISTIC REGRESSION FEATURE IMPORTANCE
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. Build feature names
# ------------------------------------------------------------
structured_feature_names = structured_feature_cols.copy()

text_feature_names = []

if USE_OPINION_TEXT:

    if TEXT_REPRESENTATION in ["tfidf", "bow"]:

        vocab_features = vectorizer.get_feature_names_out()

        home_text_feature_names = [f"home_text__{name}" for name in vocab_features]
        away_text_feature_names = [f"away_text__{name}" for name in vocab_features]

        text_feature_names = home_text_feature_names + away_text_feature_names

    elif TEXT_REPRESENTATION == "sentiment":

        sentiment_base_names = [
            "pos_count",
            "neg_count",
            "pos_minus_neg",
            "pos_rate",
            "neg_rate",
            "text_length"
        ]

        home_sent_names = [f"home_sentiment__{name}" for name in sentiment_base_names]
        away_sent_names = [f"away_sentiment__{name}" for name in sentiment_base_names]
        diff_sent_names = [f"diff_sentiment__{name}" for name in sentiment_base_names]

        text_feature_names = home_sent_names + away_sent_names + diff_sent_names

feature_names = structured_feature_names + text_feature_names

print("Number of feature names:", len(feature_names))
print("Number of model coefficients:", best_model.coef_.shape[1])

if len(feature_names) != best_model.coef_.shape[1]:
    raise ValueError("Feature names do not match model coefficient dimensions.")

# ------------------------------------------------------------
# 2. Extract coefficients
# ------------------------------------------------------------
class_names = {
    0: "Home win",
    1: "Draw",
    2: "Away win"
}

coef_df = pd.DataFrame(
    best_model.coef_,
    columns=feature_names
)

coef_df["class"] = [class_names[i] for i in best_model.classes_]

coef_long = coef_df.melt(
    id_vars="class",
    var_name="feature",
    value_name="coefficient"
)

coef_long["abs_coefficient"] = coef_long["coefficient"].abs()

# ------------------------------------------------------------
# 3. Show top features per class
# ------------------------------------------------------------
for class_label in coef_long["class"].unique():

    print("\n" + "=" * 70)
    print(f"TOP POSITIVE FEATURES FOR: {class_label}")
    print("=" * 70)

    display(
        coef_long[coef_long["class"] == class_label]
        .sort_values("coefficient", ascending=False)
        .head(20)
    )

    print("\n" + "=" * 70)
    print(f"TOP NEGATIVE FEATURES FOR: {class_label}")
    print("=" * 70)

    display(
        coef_long[coef_long["class"] == class_label]
        .sort_values("coefficient", ascending=True)
        .head(20)
    )

# ------------------------------------------------------------
# 4. Overall strongest features by absolute coefficient
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("OVERALL STRONGEST FEATURES BY ABSOLUTE COEFFICIENT")
print("=" * 70)

display(
    coef_long
    .sort_values("abs_coefficient", ascending=False)
    .head(50)
)

# ------------------------------------------------------------
# 5. Save feature importance
# ------------------------------------------------------------
importance_path = os.path.join(
    OUTPUT_DIR,
    f"{EXPERIMENT}_{TEXT_TYPE}_{MODEL_INPUT_TYPE}_lr_feature_importance.csv"
)

coef_long.to_csv(importance_path, index=False)

print("\nSaved feature importance to:")
print(importance_path)